# Nguyen et al. 2018 Mg-EDTA -- analysis

Loads the saved MCMC samples produced by `fit.py` (run it first, e.g.
`python examples/nguyen_2018_mg_edta/fit.py --quick`) and reproduces the posterior summary,
the regression comparison against the published two-component result, and a
posterior-predictive check. No sampling happens in this notebook.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

EXAMPLE_NAME = "nguyen_2018_mg_edta"
cwd = Path.cwd()
repo = cwd if cwd.name == "BayesianBinding" else next(
    (p for p in [cwd, *cwd.parents] if p.name == "BayesianBinding"), None
)
if repo is None:
    raise RuntimeError("Run this notebook from inside the BayesianBinding repository.")
sys.path.insert(0, str(repo / "src"))
sys.path.insert(0, str(repo / "examples"))
EXAMPLE_DIR = repo / "examples" / EXAMPLE_NAME
RESULTS_DIR = EXAMPLE_DIR / "results"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import _common

Load the saved samples, parameter summary, and run metadata.

In [ ]:
samples, summary, run_metadata = _common.load_results(RESULTS_DIR)
print("run:", run_metadata["mode"], run_metadata["mcmc"], f"({run_metadata['elapsed_seconds']} s)")
summary

Reproduce the published-regression comparison (95% credible intervals and posterior correlations).

In [ ]:
from bayesian_binding.regression import (
    nguyen_2018_mg_edta_regression_rows,
    summarize_mg_edta_posterior,
)

regression = summarize_mg_edta_posterior(samples)
pd.DataFrame(nguyen_2018_mg_edta_regression_rows(regression))

Posterior predictive check: observed heats with the posterior predictive mean and 95% interval.

In [ ]:
from bayesian_binding.data import load_dat

experiment = load_dat(repo / "data" / "nguyen_2018_mg_edta" / "Mg1EDTAp1a.DAT")
q = np.asarray(samples["q_model"])
mean = q.mean(axis=0)
interval = np.quantile(q, [0.025, 0.975], axis=0)
x = np.arange(1, experiment.n_injections + 1)

fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.fill_between(x, interval[0], interval[1], color="#9ecae1", alpha=0.45, label="95% posterior predictive")
ax.plot(x, mean, color="#08519c", lw=2.0, label="posterior predictive mean")
ax.scatter(x, experiment.heats_microcalorie, color="black", s=26, zorder=3, label="observed")
ax.axhline(0.0, color="0.82", lw=0.8)
ax.set_xlabel("injection"); ax.set_ylabel("heat (microcal)")
ax.set_title("Mg1EDTAp1a posterior predictive check")
ax.legend(frameon=False); fig.tight_layout();